# Bộ So Sánh 6 Mô Hình Học Chuyển Giao (Transfer Learning) & Mô Hình Đề Xuất `CNN`

Sổ tay này xây dựng, huấn luyện và đánh giá bộ **6 mô hình Học chuyển giao (Transfer Learning)** tiền huấn luyện trên ImageNet-1K theo đúng quy trình, cấu trúc tinh chỉnh tầng và hệ thống biểu đồ của bài báo tham chiếu (*"Enhancing Esophageal Cancer Detection Using Transfer Learning Techniques"* — Luan N. T. Huynh, Hoai-Phuong Nguyen-Cao et al., Thu Dau Mot University) để đối sánh trực tiếp với mô hình **`CNN`** của nhóm (`KDEF_Stem2Ch_MS4_Symmetric`) trên bộ dữ liệu khuôn mặt đa văn hóa 3 nguồn (**KDEF + RADIATE + FER-2013**, 41.269 ảnh sạch):

1. **`VGG16`** (`torchvision.models.vgg16`, `IMAGENET1K_V1`)
2. **`VGG19`** (`torchvision.models.vgg19`, `IMAGENET1K_V1`)
3. **`InceptionV3`** (`torchvision.models.inception_v3`, `IMAGENET1K_V1`)
4. **`DenseNet201`** (`torchvision.models.densenet201`, `IMAGENET1K_V1`)
5. **`ResNet50`** (`torchvision.models.resnet50`, `IMAGENET1K_V1`)
6. **`EfficientNetB4`** (`torchvision.models.efficientnet_b4`, `IMAGENET1K_V1`)
7. **`CNN` (Mô hình đề xuất của nhóm — `KDEF_Stem2Ch_MS4_Symmetric`, 1.349.165 tham số)**

---

## 1. Quy trình & Kiến trúc Học chuyển giao (Theo Mục 3.1, 3.2 & Fig. 1 bài báo tham chiếu)

- **Tiền xử lý & Tăng cường dữ liệu (Data Processing & Augmentation — Mục 3.1):**
  - Phát hiện và căn chỉnh khuôn mặt bằng SSD Square-Pad (`margin=1.25`), cân bằng tương phản cục bộ `CLAHE`, chuẩn hóa giá trị điểm ảnh về `[0, 1]` và chuẩn hóa 3 kênh RGB theo thống kê ImageNet.
  - Xử lý mất cân bằng dữ liệu bằng biến đổi hình học trên GPU: **Rotation** (xác suất `0.7`, góc xoay `±3°..5°`), **Distortion** (biến dạng lưới/affine) và **Flipping** (lật ngang ngẫu nhiên), kết hợp trọng số lớp cân bằng nghịch đảo tần suất.
- **Cấu trúc tinh chỉnh tầng phân loại (Customize Layer of Pre-trained Model — Mục 3.2 & Fig. 1):**
  - Đóng băng các tầng trích xuất đặc trưng ban đầu (*frozen initial layers*) để bảo toàn các đặc trưng hình thái cơ bản (cạnh, kết cấu, đường nét).
  - Tinh chỉnh các tầng sâu phía sau (*fine-tuned later layers*) nối với đầu phân loại tùy chỉnh:
    $$\text{Global Average Pooling} \longrightarrow \text{FC}(128) \longrightarrow \text{FC}(64) \longrightarrow \text{Dropout}(0.7) \longrightarrow \text{FC}(7, \text{Softmax})$$
  - Siêu tham số huấn luyện: tốc độ học `lr = 0.0001` (`1e-4`), huấn luyện trong **20 epochs** trên tập `train` (32.404 ảnh) và kiểm định trên tập `val` (3.990 ảnh).

![Fig. 1 Kiến trúc đề xuất](../reports/figures/paper_fig1_proposed_architecture.png)


In [1]:
import json
import sys
from pathlib import Path
import torch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.models.baselines import PAPER_TL_ARCHITECTURES, load_tl_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for arch_key, disp in PAPER_TL_ARCHITECTURES.items():
    p = REPO_ROOT / 'checkpoints' / f'{arch_key}_multicultural.pt'
    print(f'{disp:15s}: checkpoint={p.name:32s} exists={p.exists()} size={p.stat().st_size/(1024*1024):.2f} MB')

VGG16          : checkpoint=vgg16_multicultural.pt           exists=True size=56.44 MB
VGG19          : checkpoint=vgg19_multicultural.pt           exists=True size=76.70 MB
InceptionV3    : checkpoint=inception_v3_multicultural.pt    exists=True size=84.49 MB
DenseNet201    : checkpoint=densenet201_multicultural.pt     exists=True size=71.32 MB
ResNet50       : checkpoint=resnet50_multicultural.pt        exists=True size=91.04 MB
EfficientNetB4 : checkpoint=efficientnet_b4_multicultural.pt exists=True size=68.61 MB


## 2. Biểu đồ Hội tụ Accuracy & Loss qua 20 Epochs (Theo Fig. 2 & Fig. 3 bài báo tham chiếu)

![Fig. 2 Accuracy Curves](../reports/figures/paper_fig2_accuracy_curves.png)

![Fig. 3 Loss Curves](../reports/figures/paper_fig3_loss_curves.png)


In [2]:
# Kết quả đánh giá trên tập Validation (3.990 ảnh)
val_comp = json.loads((REPO_ROOT / 'results' / 'val_comparison.json').read_text(encoding='utf-8'))
paper_order = ['efficientnet_b4', 'inception_v3', 'vgg19', 'densenet201', 'vgg16', 'resnet50', 'cnn']
for k in paper_order:
    m = val_comp['models'][k]
    mt = m['metrics']
    print(f"[{m['short_name']:14s} | Val ] Acc={mt['accuracy']*100:5.2f}% | Prec={mt['macro_precision']*100:5.2f}% | Rec={mt['macro_recall']*100:5.2f}% | F1={mt['macro_f1']*100:5.2f}% | Spec={mt['macro_specificity']*100:5.2f}% | AUC={mt['roc_auc_ovr_macro']:.4f}")

[EfficientNetB4 | Val ] Acc=54.71% | Prec=52.07% | Rec=53.39% | F1=52.26% | Spec=92.28% | AUC=0.8556
[InceptionV3    | Val ] Acc=55.56% | Prec=52.85% | Rec=52.63% | F1=52.70% | Spec=92.41% | AUC=0.8330
[VGG19          | Val ] Acc=61.60% | Prec=60.49% | Rec=60.15% | F1=60.29% | Spec=93.46% | AUC=0.8776
[DenseNet201    | Val ] Acc=62.21% | Prec=61.18% | Rec=61.25% | F1=61.19% | Spec=93.56% | AUC=0.8779
[VGG16          | Val ] Acc=64.26% | Prec=62.58% | Rec=63.70% | F1=63.03% | Spec=93.91% | AUC=0.8921
[ResNet50       | Val ] Acc=65.86% | Prec=65.03% | Rec=65.20% | F1=65.09% | Spec=94.18% | AUC=0.9008
[CNN            | Val ] Acc=71.63% | Prec=71.57% | Rec=70.85% | F1=71.05% | Spec=95.16% | AUC=0.9330


In [3]:
# Bảng Table 1: So sánh hiệu năng 6 mô hình Pre-trained Transfer Learning và mô hình CNN trên tập Test khóa (4.875 ảnh)
comp = json.loads((REPO_ROOT / 'results' / 'test_comparison.json').read_text(encoding='utf-8'))
print(f"{'Model':16s} | {'Accuracy':>9s} | {'Precision':>9s} | {'Recall':>9s} | {'F1-Score':>9s} | {'Specificity':>11s} | {'ROC-AUC':>8s} | {'Params':>11s}")
print('-' * 98)
for k in paper_order:
    m = comp['models'][k]
    mt = m['metrics']
    print(f"{m['short_name']:16s} | {mt['accuracy']*100:8.2f}% | {mt['macro_precision']*100:8.2f}% | {mt['macro_recall']*100:8.2f}% | {mt['macro_f1']*100:8.2f}% | {mt['macro_specificity']*100:10.2f}% | {mt['roc_auc_ovr_macro']:8.4f} | {m['n_parameters']:11,d}")

Model            |  Accuracy | Precision |    Recall |  F1-Score | Specificity |  ROC-AUC |      Params
--------------------------------------------------------------------------------------------------
EfficientNetB4   |    55.34% |    53.15% |    53.66% |    53.09% |     92.46% |   0.8683 |  17,787,215
InceptionV3      |    57.50% |    56.14% |    56.40% |    56.21% |     92.83% |   0.8529 |  22,056,935
VGG19            |    64.47% |    64.15% |    63.84% |    63.96% |     94.00% |   0.8917 |  20,099,143
DenseNet201      |    64.51% |    63.97% |    63.65% |    63.78% |     94.01% |   0.8907 |  18,347,911
VGG16            |    66.07% |    65.52% |    65.42% |    65.44% |     94.28% |   0.9047 |  14,789,447
ResNet50         |    67.02% |    65.97% |    66.27% |    66.07% |     94.44% |   0.9098 |  23,779,399
CNN              |    73.39% |    73.81% |    72.30% |    72.82% |     95.50% |   0.9450 |   1,349,165


## 3. Biểu đồ Table 1 & Ma trận nhầm lẫn của các mô hình (Theo Table 1 & Fig. 4 bài báo tham chiếu)

![Table 1 Performance Comparison](../reports/figures/paper_table1_metrics_bar.png)

![Fig. 4 Confusion Matrices](../reports/figures/paper_fig4_confusion_matrices.png)
